Define outcomes

# Setup

In [ ]:
library(tidyverse)
library(bigrquery)
library(tictoc)
library(data.table)
library(lubridate)

In [ ]:
source("functions.R")

In [ ]:
tic("Time to run script")

# Diabetes

## Define index date and DM Type

In [ ]:
dm_df <- pull_condition("201254, 201826, 43531011, 4008576, 442793, 443238, 4016045, 43531007, 45757079",
                        name = "DiabetesMellitusTopLevel")  
dim(dm_df)

Categorize the codes into Type 1 vs. Type II/unspecified. 

In [ ]:
 #Follow the KDI style of assuming it is type 2 unless the code specifies type I
type_summary <- dm_df %>% count(standard_concept_name, condition_concept_id) %>% 
    mutate(DM_type = ifelse(grepl("type 1|type I |type I$|juvenile", standard_concept_name, ignore.case=T), 
                           "T1D", "T2D")) %>% 
    arrange(DM_type, desc(n))
options(repr.matrix.max.rows=600, repr.matrix.max.cols=200)
type_summary

In [ ]:
DMrec_out <- dm_df %>% left_join(type_summary %>% select(-n))
write_to_bucket(DMrec_out, "DM_allrecords.csv")

Summarize Diabetes codes for each person.  Get the first DM code and the number and proportion of T1D vs T2D codes. 

In [ ]:
dm_first <- dm_df %>%
    left_join_quiet(type_summary %>% ungroup() %>% select(condition_concept_id, DM_type)) %>%
    distinct(person_id, condition_start_date, source_concept_code, DM_type, .keep_all = TRUE) %>%
    group_by(person_id) %>%
    mutate(n_T1Dcode = sum(DM_type == "T1D"),
           n_T2Dcode = sum(DM_type == "T2D"),
          prop_T1Dcode = mean(DM_type == "T1D")) %>%
    slice_min(condition_start_date) %>%
    ungroup() %>%
    distinct(person_id, .keep_all = TRUE) %>%
    select(-DM_type) %>%
    dplyr::rename(DMDx_first = condition_start_date)
nrow(dm_first)

For Type II DM, they need to have 2+ codes at least 30 days apart. 

In [ ]:
T2D_days <- dm_df %>%
    left_join_quiet(type_summary %>% ungroup() %>% select(condition_concept_id, DM_type)) %>%
    filter(DM_type == "T2D") %>%
    group_by(person_id) %>%
    summarize(days_dif_t2d = as.numeric(difftime(max(condition_start_date), 
                                                 min(condition_start_date), 
                                                 units = "days"))) %>%
    ungroup() 
              
table(T2D_days$days_dif_t2d >= 30)

Get insulin and glucagon info for the klompas algorithm.
For T1D, patients must use insulin at some point.  They must also either use glucagon or NOT use oral dm meds (except metformin which is allowed), in addition to DX-code-based requirements for T1D,

In [ ]:
covars <- read_from_bucket("covariates_wide_all_participants.csv", skip_copy = T)
insulin <- read_from_bucket("insulin.csv", skip_copy=T) %>%
    mutate(drug_exposure_start_datetime = as.Date(drug_exposure_start_datetime))
oraldm_drug <- read_from_bucket("oral_dmdrug_excl_insulin_metformin.csv", skip_copy=T)  %>%
    mutate(drug_exposure_start_datetime = as.Date(drug_exposure_start_datetime))
glucagon <- read_from_bucket("glucagon.csv", skip_copy=T)  %>%
    mutate(drug_exposure_start_datetime = as.Date(drug_exposure_start_datetime))

In [ ]:
first_insulin <- insulin %>% 
    group_by(person_id) %>% 
    slice_min(drug_exposure_start_datetime) %>%
    distinct(drug_exposure_start_datetime) %>% 
    dplyr::rename(first_insulin = drug_exposure_start_datetime)
first_oraldm_drug <- oraldm_drug %>%
    group_by(person_id) %>%
    slice_min(drug_exposure_start_datetime) %>%
    distinct(drug_exposure_start_datetime) %>%
    dplyr::rename(first_oraldm_drug = drug_exposure_start_datetime)
first_glucagon <- glucagon %>%
    group_by(person_id) %>%
    slice_min(drug_exposure_start_datetime) %>%
    distinct(drug_exposure_start_datetime) %>%
    dplyr::rename(first_glucagon = drug_exposure_start_datetime)

In [ ]:
dm_first <- dm_first %>%
    left_join_quiet(first_insulin) %>%
    left_join_quiet(first_oraldm_drug) %>%
    left_join_quiet(first_glucagon) %>%
    left_join_quiet(T2D_days) %>%
    mutate(DM_type = case_when(n_T1Dcode >= 2 & prop_T1Dcode >= 0.5 & !is.na(first_insulin) &
                                        (!is.na(first_glucagon) | is.na(first_oraldm_drug)) ~ "T1D",
                               n_T2Dcode >= 2 & days_dif_t2d >= 30 ~ "T2D", 
                               TRUE ~ "Not categorized"))

table(dm_first$DM_type)
table(dm_first$DM_type, dm_first$n_T2Dcode >= 2)

Move the date of first DmDx backwards in time if they took insulin or oral DM drugs prior to the first appearance of a DM Dx code. 

In [ ]:
dm_first %>% count(first_insulin < DMDx_first)
dm_first %>% count(first_oraldm_drug < DMDx_first)

In [ ]:
dm_first <- dm_first %>%
    mutate(DMDx_first = pmin(DMDx_first, first_insulin, first_oraldm_drug, na.rm=T))

## Define a Flag for Newly Diagnosed DM based on primary care usage in prior 2 years

Next, we want to define a flag for whether the DM patient had outpatient or home visits in both of the 2 yrs prior to their DmDx. 

In [ ]:
all_outpat_home_visits_df <- read_from_bucket('all_outpatient_and_home_visit_dates.csv', skip_copy = TRUE)
gc()

In [ ]:
DM_outpat_visits_pre_DM <- inner_join_quiet(all_outpat_home_visits_df, 
                                      dm_first %>% select(person_id, DMDx_first)) %>%
    filter(date_outpat_or_home_visit < DMDx_first) %>%
    # simplify to rounded year before DM, distinct
    mutate(years_before_DM = round(decimal_date(date_outpat_or_home_visit) - decimal_date(DMDx_first), 0)) %>%
    filter(years_before_DM >= -2 & years_before_DM < 0) %>%
    distinct(person_id, years_before_DM)

#These patients meet the criteria
DM_p2 <- DM_outpat_visits_pre_DM %>%
    group_by(person_id) %>%
    filter(sum(years_before_DM) == -3) %>%
    distinct(person_id) %>% 
    mutate(DM_prior2yr_outpat = 1)
nrow(DM_p2)

In [ ]:
DM_summary <- dm_first %>%
    select(person_id, DMDx_first, DM_type) %>%
    left_join(DM_p2) %>%
    mutate(DM_prior2yr_outpat = coalesce(DM_prior2yr_outpat, 0))

# Renal disease

## Kidney disease from Clinical Codes

### ESRD and related fields

#### Dialysis

##### Concepts

In [ ]:
dialysis_proc_codes <- "2213573, 4247794, 4197217, 4289454, 2213576, 4026915, 4324124, 
    2213577, 2617342, 40757115, 4224193, 4146536, 4183419, 4032243, 
    42627928, 42627979, 42628018, 42628019, 42628057, 42628058, 42628575, 42628576, 42628580, 
    2213572, 42739840, 2213575, 2213598, 42739891, 2721479, 2721482, 2788041, 43018323" 


dialysis_obs_codes <- "2617401, 2617544, 2617545, 313232, 4019967, 40480136, 40482357, 
    40483083, 4059475, 40664693, 4090651, 4092504, 4177685, 4181476, 4204620, 4301680, 
    46270032, 765776, 2101833, 2101834, 2108564, 2108566, 2108567, 2108568, 
    2213587, 2213589, 2213590, 2213592, 2213593, 2213594, 2213595,  2213601, 
    2213588, 2213591, 2213596, 2213599, 42738339, 42738678, 42738679, 2213578, 2213579, 
    2213580, 2213581, 2213582, 2213583, 2213584, 2213585, 2213586, 2213597, 2514586, 
    40664745, 40664909, 43533281, 44786445, 44786446, 44786469, 44786470, 44786471"

dialysis_cond_codes <- "4030520,45769906,45757392,43021864,45772751,45757393,44782717,601166,
                        45769904,45757392,44782717,43021864"

dialysis_device_codes <- "2614998, 2614999, 4281167"

dialysis_clinical_codes <- paste(dialysis_proc_codes, dialysis_obs_codes, dialysis_cond_codes, sep = ", ") 

In [ ]:
#get additional codes via descendants:
desc_dialysis_clinical_concepts <- getUniqueDescendantsMulti(dialysis_clinical_codes)
desc_dialysis_clinical_concepts <-  paste(desc_dialysis_clinical_concepts$concept_id, collapse = ", ")
desc_dialysis_clinical_concepts

In [ ]:
dialysis_clinical_codes <- paste(dialysis_clinical_codes, desc_dialysis_clinical_concepts, sep = ", ")

##### Dialysis PROCEDURES DOMAIN

We maintain consistency with the previous version by allowing non-standard concepts to be considered for dialysis and kidney transplantation. All concept names are displayed and reviewed. 

In [ ]:
dialysis_proc <- pull_procedure(dialysis_clinical_codes,  #dialysis_proc_codes, 
                                name = "dialysis_proc", require_standard = FALSE)
dim(dialysis_proc)

In [ ]:
dialysis_proc %>% count(procedure_concept_id, standard_concept_name, sort=T)

In [ ]:
dialysis_proc <- dialysis_proc %>% 
                    filter(!(procedure_concept_id %in% 
                             c(4080968, 4052536, 1531632, 1524104, 1531630, 1531631, 2108165)))
nrow(dialysis_proc)

##### Dialysis OBSERVATION DOMAIN

In [ ]:
Dialysis_observation_df <- pull_observation(dialysis_clinical_codes, 
                                            name = "Dialysis")
dim(Dialysis_observation_df)

In [ ]:
Dialysis_observation_df %>% count(observation_concept_id, standard_concept_name, sort=T) 

In [ ]:
Dialysis_observation_df %>% count(source_vocabulary, source_concept_code, source_concept_name, sort=T) 

##### Dialysis CONDITIONS DOMAIN

In [ ]:
dialysis_cond_df <- pull_condition(dialysis_clinical_codes, name = "dialysis_cond") 
dim(dialysis_cond_df)

In [ ]:
dialysis_cond_df %>% count(condition_concept_id, standard_concept_name, sort=T) 

##### Dialysis DEVICE DOMAIN

In [ ]:
dialysis_device_sql <- paste("
    SELECT
        device.person_id,
        device.device_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        device.device_exposure_start_datetime,
        device.device_exposure_end_datetime,
        device.device_type_concept_id,
        d_type.concept_name as device_type_concept_name,
        device.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        device.device_source_value,
        device.device_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `device_exposure` device 
        WHERE
            (
                device_concept_id IN (",  dialysis_device_codes, ")
            )) device 
    LEFT JOIN
        `concept` d_standard_concept 
            ON device.device_concept_id = d_standard_concept.concept_id 
    LEFT JOIN
        `concept` d_type 
            ON device.device_type_concept_id = d_type.concept_id 
    LEFT JOIN
        `visit_occurrence` v 
            ON device.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `concept` d_source_concept 
            ON device.device_source_concept_id = d_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
dialysis_device_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
#   strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "device_96431095",
  "dialysis_device_*.csv")

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dialysis_device_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  dialysis_device_path,
  destination_format = "CSV")


read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), device_type_concept_name = col_character(), visit_occurrence_concept_name = col_character(), device_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

dialysis_device_df <- read_bq_export_from_workspace_bucket(dialysis_device_path) %>%
     mutate(observation_date = as.Date(device_exposure_start_datetime))

dim(dialysis_device_df)

In [ ]:
dialysis_device_df %>% count(device_concept_id, standard_concept_name, sort=T) 

In [ ]:
dialysis_device_df %>% count(source_concept_code, source_concept_name, sort=T) 

##### Calculate first Dialysis

In [ ]:
Dialysis_all <- dialysis_proc %>% 
        dplyr::rename(concept_id = procedure_concept_id, event_date = procedure_date) %>%
        mutate(event_domain = "Procedure") %>%
    full_join_quiet(Dialysis_observation_df %>% 
        dplyr::rename(concept_id = observation_concept_id, event_date = observation_date) %>%
        mutate(event_domain = "Observation")) %>%
    full_join(dialysis_cond_df %>% 
        dplyr::rename(concept_id = condition_concept_id, event_date = condition_start_date) %>%
        mutate(event_domain = "Condition")) %>%
    full_join_quiet(dialysis_device_df %>% 
        dplyr::rename(concept_id = device_concept_id, event_date = observation_date) %>%
        mutate(event_domain = "Device")) %>%
    distinct()

In [ ]:
Dialysis_first <- Dialysis_all %>%
    group_by(person_id) %>%
    mutate(n_dialysis_dates = length(unique(event_date)),
          n_dialysis_domains = length(unique(event_domain))) %>%
    slice_min(event_date) %>%
    summarize(Dialysis_first = min(event_date), 
             standard_concept_name = paste0(unique(standard_concept_name), collapse = " / "),
             event_domain = paste0(unique(event_domain), collapse = " / "),
             n_dialysis_dates = unique(n_dialysis_dates),
             n_dialysis_domains = unique(n_dialysis_domains)) 
nrow(Dialysis_first)

#### Kidney transplantation

##### Kidney transplantation concepts

In [ ]:
xplant_proc_codes <- "2211744, 2109586, 2101033, 42737578, 4322471, 2792053, 2109587, 
    2109588, 2774517, 2003626, 2774520, 4022805, 4197300, 4021107, 2774521, 2774518, 
    2109580, 2109581, 2109582, 2109583, 2109584, 2109585,  2721092, 2774519, 2774522" 
KidneyTransplant_observation_codes <- "4059356, 38001396, 38000894"
xplant_cond_codes <- "199991, 42539502, 4127554, 4128369, 4324887" 

KidneyTransplant_clinical_codes <- paste(xplant_proc_codes, KidneyTransplant_observation_codes, 
                                          xplant_cond_codes, sep = ", ")

In [ ]:
addl_xplant_codes <- getUniqueDescendantsMulti(KidneyTransplant_clinical_codes)
addl_xplant_codes <- paste(addl_xplant_codes$concept_id, collapse = ", ")
addl_xplant_codes

In [ ]:
KidneyTransplant_clinical_codes <- paste(KidneyTransplant_clinical_codes, addl_xplant_codes, sep = ", ")

In [ ]:
KidneyTransplant_clinical_codes

##### Kidney transplantation PROCEDURES DOMAIN

In [ ]:
Kidney_xplant <- pull_procedure(KidneyTransplant_clinical_codes, 
                        name = "kidney_transplant_nonstandard", require_standard = FALSE)                              
dim(Kidney_xplant)

In [ ]:
Kidney_xplant %>% count(procedure_concept_id, standard_concept_name, sort=T) 

In [ ]:
Kidney_xplant <- Kidney_xplant %>% filter(procedure_concept_id != 4022474)

In [ ]:
#Check these results against the survey results
kxs <- read_cols("covariates_wide_all_participants.csv", 
                        select = c("person_id", "KidneyTransplant_survey", "survey_first_datetime"), skip_copy = TRUE)

In [ ]:
Kidney_xplant %>% 
    group_by(person_id) %>%
    slice_min(procedure_date) %>%
    distinct(person_id, .keep_all = TRUE) %>%
    left_join(kxs) %>%
    filter(procedure_date <= survey_first_datetime) %>%
    group_by(procedure_concept_id, standard_concept_name) %>%
    summarize(n = n(), percent_selfreport = mean(KidneyTransplant_survey, na.rm = T))

Good concordance

##### Kidney Transplantation OBSERVATION DOMAIN

In [ ]:
KidneyTransplant_observation_df <- pull_observation(KidneyTransplant_clinical_codes,
                                                    name="KidneyTransplant")
dim(KidneyTransplant_observation_df)

In [ ]:
KidneyTransplant_observation_df %>% count(observation_concept_id, standard_concept_name, sort=T) 

In [ ]:
KidneyTransplant_observation_df %>% 
    group_by(person_id) %>%
    slice_min(observation_date) %>%
    distinct(person_id, .keep_all = TRUE) %>%
    left_join(kxs) %>%
    filter(observation_date <= survey_first_datetime) %>%
    group_by(observation_concept_id, standard_concept_name) %>%
    summarize(n = n(), percent_selfreport = mean(KidneyTransplant_survey, na.rm = T))

Good agreement vs. the self-report

##### Kidney Transplantation CONDITIONS DOMAIN

In [ ]:
xplant_cond_df <- pull_condition(KidneyTransplant_clinical_codes,
                                 name = "xplant_cond") 
dim(xplant_cond_df)

In [ ]:
xplant_cond_df %>% count(condition_concept_id, standard_concept_name, sort=T) 

##### Calculate first Kidney transplantation

In [ ]:
Kidney_xplant_first <- Kidney_xplant %>%
        dplyr::rename(concept_id = procedure_concept_id, event_date = procedure_date) %>%
        mutate(event_domain = "Procedure") %>%
    full_join_quiet(KidneyTransplant_observation_df %>% 
                    dplyr::rename(concept_id = observation_concept_id, event_date = observation_date) %>%
                    mutate(event_domain = "Observation")) %>%
    full_join_quiet(xplant_cond_df %>% 
                    dplyr::rename(concept_id = condition_concept_id, event_date = condition_start_date) %>%
                    mutate(event_domain = "Condition")) %>%
    group_by(person_id) %>%
    slice_min(event_date) %>%
    summarize(Kidney_xplant_first = min(event_date),
             standard_concept_name = paste0(unique(standard_concept_name), collapse = " / "),
             event_domain = paste0(unique(event_domain), collapse = " / ")) 

#### Dx Codes for ESRD / Renal Failure

In [ ]:
# Those for dialysis/transplant will also be added
renalfailure_codes <- "43021247, 4027133, 4324887, 198185, 44782924, 4059129, 193782, 443919, 43020455"
RenalFailureDx_df <- pull_condition(renalfailure_codes,
                                  name = "RenalFailureDxTopLevel")
dim(RenalFailureDx_df)

#### Dx Codes for CKD stage 5

In [ ]:
ckd5_codes <- "44782690, 443611, 43020455, 44784439, 439694, 439695, 
    46270356, 46273164, 44784638, 44782717, 45757393, 43020437, 
    45757392, 43021864, 762973, 37018886, 45772751, 439694, 44784638"
CKD5_df <- pull_condition(ckd5_codes, name = "CKD5")                       
dim(CKD5_df)

In [ ]:
CKD5_df %>% count(condition_concept_id, standard_concept_name, sort=T) 

In [ ]:
CKD5_df %>% count(source_concept_code, source_concept_name, sort=T)

In [ ]:
#need to filter out these source codes:
#403.90	Hypertensive chronic kidney disease, unspecified, with chronic kidney disease stage I through stage IV, or unspecified
#403.10	Hypertensive chronic kidney disease, benign, with chronic kidney disease stage I through stage IV, or unspecified

In [ ]:
CKD5_df <- CKD5_df %>% 
    filter(!( !is.na(CKD5_df$source_concept_code) & CKD5_df$source_concept_code %in% c("403.90", "403.10")))

#### Combine codes eligible for defining renal failure seq outcome

Includes Renal Failure, CKD5, Dialysis, Transplant. Apply `code_seqence` to ensure codes appear within day range.

Will define final `RenalFailure_first` variable below when data from biomarkers are included

In [ ]:
renal_failure_allcodes <- rbind(
    # bind kidney failure 
    RenalFailureDx_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "esrd") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    # bind all dialysis
    dialysis_proc %>% 
        select(person_id, procedure_date, procedure_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "dialysis") %>%
        rename(event_date = procedure_date,
              concept_id = procedure_concept_id),
    Dialysis_observation_df %>% 
        select(person_id, observation_date, observation_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "dialysis") %>%
        rename(event_date = observation_date,
              concept_id = observation_concept_id),
    dialysis_cond_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "dialysis") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    dialysis_device_df %>% 
        select(person_id, observation_date, device_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "dialysis") %>%
        rename(event_date = observation_date,
              concept_id = device_concept_id),
    # bind all transplant
    Kidney_xplant %>% 
        select(person_id, procedure_date, procedure_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "transplant") %>%
        rename(event_date = procedure_date,
              concept_id = procedure_concept_id),
    KidneyTransplant_observation_df %>% 
        select(person_id, observation_date, observation_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "transplant") %>%
        rename(event_date = observation_date,
              concept_id = observation_concept_id),
    xplant_cond_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "transplant") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    # bind ckd5 
    CKD5_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name,
              source_vocabulary, source_concept_code, source_concept_name) %>%
        mutate(code_type = "ckd5") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id)
) %>% 
    distinct()


In [ ]:
#review source codes
renal_failure_allcodes %>% count(source_concept_code, source_concept_name, source_vocabulary, sort=T)

In [ ]:
renal_failure_allcodes <- renal_failure_allcodes %>%
    filter(!(source_concept_code %in% c('403.90', '403.10', 'D63.1', 'N17.0', 'N17.9', 'N18.30', 'N18.32', 'N18.3', 'N18.9',
                                       '90688005', # is a high-level ancestor with descendants of all stages
                                        '236433006', # not appropriate
                                        'G0052'))) # is not corroborated by egfr

In [ ]:
write_to_bucket(renal_failure_allcodes, "renal_failure_allcodes.csv")

In [ ]:
RenalFailure_first <- code_sequence(
    renal_failure_allcodes %>% distinct(person_id, event_date), 
    45, 365, 2, prefix = "RenalFailure"
)

### CKD stages 3a, 3b, 4, and any CKD

#### CKD4
Can be less severe than renal failure, but also includes renal failure.  

In [ ]:
# Extract Stage 4 disease ONLY
ckd4_codes <- "443612, 195314, 44784639, 43531577, 44782689, 37398911"
CKD4_df <- pull_condition(ckd4_codes, name = "CKD4")

In [ ]:
CKD4_df %>% count(condition_concept_id, standard_concept_name, sort=T) 

In [ ]:
CKD4_allcodes <- bind_rows(
    # bind ckd4
    CKD4_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name) %>%
        mutate(code_type = "ckd4") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    # bind all rf
    renal_failure_allcodes
    )

In [ ]:
CKD4_first <- code_sequence(CKD4_allcodes, 90, 365, 2, prefix = "CKD4")
nrow(CKD4_first)

In [ ]:
CKD4_first <- CKD4_first %>%
    full_join_quiet(RenalFailure_first) %>%
    mutate(first_CKD4.2.90.365 = pmin(first_CKD4.2.90.365, first_RenalFailure.2.45.365, na.rm=T)) %>%
    distinct(person_id, first_CKD4.2.90.365)

#### CKD3b

In [ ]:
ckd3b_codes <- "45763855"
CKD3b_df <- pull_condition(ckd3b_codes, name = "CKD3b")                         
dim(CKD3b_df)

In [ ]:
CKD3b_df %>% count(condition_concept_id, standard_concept_name, sort=T)

In [ ]:
CKD3b_allcodes <- bind_rows(
    # bind 
    CKD3b_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name) %>%
        mutate(code_type = "ckd3b") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    # bind all rf
    CKD4_allcodes
    )

In [ ]:
CKD3b_first <- code_sequence(CKD3b_allcodes, 90, 365, 2, prefix = "CKD3b")

In [ ]:
CKD3b_first <- CKD3b_first %>%
    full_join_quiet(CKD4_first) %>%
    mutate(first_CKD3b.2.90.365 = pmin(first_CKD3b.2.90.365, first_CKD4.2.90.365, na.rm=T)) %>%
    distinct(person_id, first_CKD3b.2.90.365)

#### CKD3a

In [ ]:
#Includes 3a, 3b, unspecified stage 3 (a/b)
ckd3a_codes <- "443597, 45763854, 43531653, 37019193, 44782691, 45771075, 43021835, 46273636, 45757446"    
CKD3a_df <- pull_condition(ckd3a_codes, name = "CKD3a")                         
dim(CKD3a_df)

In [ ]:
CKD3a_df %>% count(condition_concept_id, standard_concept_name, sort=T)

In [ ]:
CKD3a_allcodes <- bind_rows(
    CKD3a_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name) %>%
        mutate(code_type = "ckd3a") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    # bind all rf
    CKD3b_allcodes
    )

In [ ]:
CKD3a_first <- code_sequence(CKD3a_allcodes, 
                             90, 365, 2, prefix = "CKD3a")

In [ ]:
CKD3a_first <- CKD3a_first %>%
    full_join_quiet(CKD3b_first) %>%
    mutate(first_CKD3a.2.90.365 = pmin(first_CKD3a.2.90.365, first_CKD3b.2.90.365, na.rm=T)) %>%
    distinct(person_id, first_CKD3a.2.90.365)

#### CKD, any stage

In [ ]:
ckd_unstaged <- "192279, 200687, 36716947, 37017533, 37017813, 37018886, 4027133, 
    4030520, 4032298, 40482458, 40486245, 4059129, 4070976, 4081057, 4125970, 4126451, 
    4128370, 4128371, 4128372, 4206115, 43021247, 43021389, 43021418, 43021985, 4324887, 
    43531562, 43531577, 43531578, 435649, 438624, 440302, 443212, 443961, 444024, 44782689, 
    44784638, 45757499, 45768813, 45769906, 765054, 443612, 4137220, 44782429, 44784621, 
    443919, 439696, 443597, 45763854, 45763855, 46271022, 443614, 443601, 195314, 443238, 
    40484648, 44784621, 44782429, 45768812, 133810, 46270426, 443731, 4035437"

#combine condition codes
ckd_any <- paste(ckd_unstaged, ckd3a_codes, ckd3b_codes, ckd4_codes, ckd5_codes,
                 renalfailure_codes, xplant_cond_codes, sep = ", ")

In [ ]:
CKD_any_df <- pull_condition(ckd_any, name = "CKD_any")
dim(CKD_any_df)

In [ ]:
CKD_any_df %>% count(condition_concept_id, standard_concept_name, sort=T)

In [ ]:
CKD_any_allcodes <- bind_rows(
    CKD_any_df %>% 
        select(person_id, condition_start_date, condition_concept_id, standard_concept_name) %>%
        mutate(code_type = "ckd_any") %>%
        rename(event_date = condition_start_date,
              concept_id = condition_concept_id),
    CKD3a_allcodes #adds in all prior codes including procedure/device/observation
    ) %>% distinct()

In [ ]:
CKD_first <- code_sequence(CKD_any_allcodes, 90, 365, 2, prefix = "CKD")

In [ ]:
CKD_first <- CKD_first %>%
    full_join_quiet(CKD3a_first) %>%
    mutate(first_CKD.2.90.365 = pmin(first_CKD3a.2.90.365, first_CKD.2.90.365, na.rm=T)) %>%
    distinct(person_id, first_CKD.2.90.365)

In [ ]:
nrow(CKD_first)
nrow(CKD3a_first)
nrow(CKD3b_first)
nrow(CKD4_first)
nrow(RenalFailure_first)

## Kidney disease from biomarkers (eGFR, UACR)

### Albuminuria

In [ ]:
fullcohortIDs <- read_cols("covariates_wide_all_participants.csv", select = "person_id", skip_copy = TRUE)

In [ ]:
UACR <- read_cols("bm_UACR.csv", select = c("person_id", "measurement_date", "UACR"), skip_copy=T) %>% 
    mutate(measurement_date = as.Date(measurement_date)) 

In [ ]:
macroalbuminuria_first <- UACR %>%
    filter(UACR >= 300) %>%
    mutate(measurement_date = as.Date(measurement_date)) %>% 
    group_by(person_id) %>%
    slice_min(measurement_date) %>%
    dplyr::rename(macroalbuminuria_first = measurement_date) %>%
    dplyr::select(person_id, macroalbuminuria_first)
    
nrow(macroalbuminuria_first)

In [ ]:
microalbuminuria_first <- UACR %>%
    filter(UACR >= 30) %>%
    mutate(measurement_date = as.Date(measurement_date)) %>% 
    group_by(person_id) %>%
    slice_min(measurement_date) %>%
    dplyr::rename(microalbuminuria_first = measurement_date) %>%
    dplyr::select(person_id, microalbuminuria_first)
    
nrow(microalbuminuria_first)

### Persistent low eGFR

In [ ]:
eGFR <- read_cols("bm_eGFR_creatinine.csv", select = c("person_id", "measurement_date", "eGFR"), skip_copy=T) %>% 
    mutate(measurement_date = as.Date(measurement_date)) 

In [ ]:
EGFR_threshold <- 15
EGFR_days_threshold <- 45

prolonged_low_egfr_firstoccur_15_45 <- eGFR %>%
    group_by(person_id) %>%
    filter(any(eGFR <= EGFR_threshold)) %>%
    arrange(person_id, measurement_date) %>%
    mutate(EGFRlt = eGFR <= EGFR_threshold,
          nextEGFRlt = lead(EGFRlt), 
          previousEGFRlt = lag(EGFRlt)) %>%
    filter( !(EGFRlt == nextEGFRlt  &  EGFRlt == previousEGFRlt) | is.na(previousEGFRlt) | is.na(nextEGFRlt)) %>%
    mutate(days_between = difftime(lead(measurement_date), measurement_date, units = "days")) %>%
    filter(EGFRlt & nextEGFRlt & days_between >= EGFR_days_threshold) %>%
    group_by(person_id) %>%
    summarize(plEGFR_15_45_first = min(measurement_date))

nrow(prolonged_low_egfr_firstoccur_15_45)

In [ ]:
EGFR_threshold <- 30
EGFR_days_threshold <- 90

prolonged_low_egfr_firstoccur_30_90 <- eGFR %>%
    group_by(person_id) %>%
    filter(any(eGFR <= EGFR_threshold)) %>%
    arrange(person_id, measurement_date) %>%
    mutate(EGFRlt = eGFR <= EGFR_threshold,
          nextEGFRlt = lead(EGFRlt), 
          previousEGFRlt = lag(EGFRlt)) %>%
    filter( !(EGFRlt == nextEGFRlt  &  EGFRlt == previousEGFRlt) | is.na(previousEGFRlt) | is.na(nextEGFRlt)) %>%
    mutate(days_between = difftime(lead(measurement_date), measurement_date, units = "days")) %>%
    filter(EGFRlt & nextEGFRlt & days_between >= EGFR_days_threshold) %>%
    group_by(person_id) %>%
    summarize(plEGFR_30_90_first = min(measurement_date))

nrow(prolonged_low_egfr_firstoccur_30_90)

In [ ]:
EGFR_threshold <- 45
EGFR_days_threshold <- 90

prolonged_low_egfr_firstoccur_45_90 <- eGFR %>%
    group_by(person_id) %>%
    filter(any(eGFR <= EGFR_threshold)) %>%
    arrange(person_id, measurement_date) %>%
    mutate(EGFRlt = eGFR <= EGFR_threshold,
          nextEGFRlt = lead(EGFRlt), 
          previousEGFRlt = lag(EGFRlt)) %>%
    filter( !(EGFRlt == nextEGFRlt  &  EGFRlt == previousEGFRlt) | is.na(previousEGFRlt) | is.na(nextEGFRlt)) %>%
    mutate(days_between = difftime(lead(measurement_date), measurement_date, units = "days")) %>%
    filter(EGFRlt & nextEGFRlt & days_between >= EGFR_days_threshold) %>%
    group_by(person_id) %>%
    summarize(plEGFR_45_90_first = min(measurement_date))

nrow(prolonged_low_egfr_firstoccur_45_90)

In [ ]:
EGFR_threshold <- 60
EGFR_days_threshold <- 90

prolonged_low_egfr_firstoccur_60_90 <- eGFR %>%
    group_by(person_id) %>%
    filter(any(eGFR <= EGFR_threshold)) %>%
    arrange(person_id, measurement_date) %>%
    mutate(EGFRlt = eGFR <= EGFR_threshold,
          nextEGFRlt = lead(EGFRlt), 
          previousEGFRlt = lag(EGFRlt)) %>%
    filter( !(EGFRlt == nextEGFRlt  &  EGFRlt == previousEGFRlt) | is.na(previousEGFRlt) | is.na(nextEGFRlt)) %>%
    mutate(days_between = difftime(lead(measurement_date), measurement_date, units = "days")) %>%
    filter(EGFRlt & nextEGFRlt & days_between >= EGFR_days_threshold) %>%
    group_by(person_id) %>%
    summarize(plEGFR_60_90_first = min(measurement_date))

nrow(prolonged_low_egfr_firstoccur_60_90)

## Combine Clinical Codes + Biomarkers, Define variables 

Using output from first outcomes using sequence in above sections, as well as biomarkers.

In [ ]:
kidney_disease <- fullcohortIDs %>% 
    left_join_quiet(Dialysis_first %>% select(-standard_concept_name, -event_domain, 
                                              -n_dialysis_dates, -n_dialysis_domains)) %>%
    left_join_quiet(Kidney_xplant_first %>% select(-standard_concept_name, -event_domain)) %>%
    left_join_quiet(RenalFailure_first) %>%
    
    left_join_quiet(CKD4_first) %>%
    left_join_quiet(CKD3b_first) %>%
    left_join_quiet(CKD3a_first) %>%
    left_join_quiet(CKD_first) %>%
    
    left_join_quiet(prolonged_low_egfr_firstoccur_30_90) %>%
    left_join_quiet(prolonged_low_egfr_firstoccur_15_45) %>%
    left_join_quiet(prolonged_low_egfr_firstoccur_45_90) %>%
    left_join_quiet(prolonged_low_egfr_firstoccur_60_90) %>%
    left_join_quiet(macroalbuminuria_first) %>%
    left_join_quiet(microalbuminuria_first) %>%
    ungroup() 

In [ ]:
kidney_disease <- kidney_disease %>%
    mutate(RenalFailure_first = pmin(first_RenalFailure.2.45.365, 
                                     plEGFR_15_45_first, 
                                     na.rm=T),
           RenalFailure = as.numeric(!is.na(RenalFailure_first))) %>%
    mutate(CKD4_first = pmin(first_CKD4.2.90.365, 
                             plEGFR_30_90_first, 
                             RenalFailure_first, 
                             na.rm=T),
           CKD4 = as.numeric(!is.na(CKD4_first))) %>%
    mutate(CKD3b_first = pmin(first_CKD3b.2.90.365, 
                             plEGFR_45_90_first,
                             CKD4_first, 
                             na.rm=T),
           CKD3b = as.numeric(!is.na(CKD3b_first))) %>%
    mutate(CKD3a_first = pmin(first_CKD3a.2.90.365, 
                             plEGFR_60_90_first,
                             CKD3b_first, 
                             na.rm=T),
           CKD3a = as.numeric(!is.na(CKD3a_first))) %>%

    mutate(CKD_first = pmin(first_CKD.2.90.365, 
                             plEGFR_60_90_first,
                             CKD3a_first, 
                             na.rm=T),
           CKD = as.numeric(!is.na(CKD_first))) %>%
    mutate(Kidney_xplant = as.numeric(!is.na(Kidney_xplant_first)),
           Dialysis = as.numeric(!is.na(Dialysis_first))) %>%
    distinct() %>%
    left_join(kxs)

In [ ]:
AnyRFsumm <- renal_failure_allcodes %>%
    group_by(person_id) %>%
    summarize(AnyRF_n_dates = length(unique(event_date)),
             AnyRF_n_records = n(),
             AnyRF_n_types = length(unique(code_type)),
             AnyRF_date_range = max(event_date) - min(event_date)) %>%
    mutate(AnyRF = 1)

In [ ]:
kidney_disease <- left_join(kidney_disease, AnyRFsumm)

Get a basic summary of kidney disease among the Diabetes patients. 

In [ ]:
kd_dm <- left_join(DM_summary, kidney_disease)

table(kd_dm$CKD4)
mean(kd_dm$CKD4, na.rm=T)
table(kd_dm$RenalFailure)
mean(kd_dm$RenalFailure)
table(kd_dm$CKD4, kd_dm$RenalFailure)

# CVD

## Myocardial Infarction

In [ ]:
Myocardial_Infarction_df <- pull_condition("4329847", name = "Myocardial_Infarction")
dim(Myocardial_Infarction_df)

In [ ]:
Myocardial_Infarction_df %>% count(standard_concept_name, sort=T) 

In [ ]:
MI <- Myocardial_Infarction_df %>%
    group_by(person_id) %>%
    summarize(MI_first = min(as.Date(condition_start_date)))
nrow(MI)

## Ischemic Stroke

Use concepts from S2 of https://doi.org/10.1371%2Fjournal.pone.0226718. 

In [ ]:
#These concepts should have their descendants included. 
Ischemic_Stroke_concepts <- c(443454, 4043731, 4045737, 4046360, 4108356, 4110189, 4110190, 4110192,4111714,4159140, 43531607, 44782773, 45767658,
                              45772786, 46270031, 46270380, 46270381, 46273649)

#These concepts should NOT have their descendants included. 
Ischemic_Stroke_SOLO_concepts <- c(761110, 762933, 762934, 762935,762937, 762951, 763015, 765515, 4045735, 4045738, 4046237, 
                              4046358, 4046359,4046361, 4077086, 4111711, 4119140, 4131383, 4138327,
                              4141405, 4142739, 4145897, 4146185, 4153352,  4211509, 4319146, 35610084,
                              35610085, 36717605, 37110678, 37110679, 43530683)
                                                  
Ischemic_Stroke_string <- paste0(Ischemic_Stroke_concepts, collapse = ", ")
Ischemic_Stroke_SOLO_string <- paste0(Ischemic_Stroke_SOLO_concepts, collapse = ", ")

In [ ]:
# pull 1
Ischemic_Stroke_df <- pull_condition(Ischemic_Stroke_string, name = "IschStrokeOHDSI_1")
dim(Ischemic_Stroke_df)

In [ ]:
# pull 2
Ischemic_Stroke_df2 <- pull_condition_without_descendants(Ischemic_Stroke_SOLO_string, , name = "IschStrokeOHDSI_2")
dim(Ischemic_Stroke_df2)

In [ ]:
Ischemic_Stroke <- full_join(distinct(Ischemic_Stroke_df), distinct(Ischemic_Stroke_df2))
nrow(Ischemic_Stroke)

In [ ]:
Ischemic_Stroke %>% count(condition_concept_id, standard_concept_name, sort=T) 

In [ ]:
Ischemic_Stroke <- Ischemic_Stroke %>% 
    group_by(person_id) %>%
    summarize(Ischemic_Stroke_first = min(as.Date(condition_start_date)))
nrow(Ischemic_Stroke)

## Heart Failure

In [ ]:
Heart_Failure_df <- pull_condition("316139", name = "HeartFailure")
dim(Heart_Failure_df)

In [ ]:
Heart_Failure <- Heart_Failure_df %>% 
    group_by(person_id) %>%
    summarize(Heart_Failure_first = min(as.Date(condition_start_date)),
             standard_concept_name = paste0(standard_concept_name, collapse = " / "))
nrow(Heart_Failure)

In [ ]:
Heart_Failure %>% count(standard_concept_name, sort=T) %>%
    filter(!grepl("/", standard_concept_name)) 

## Define CVD composite variables

In [ ]:
CVD_summary <- left_join(fullcohortIDs, MI %>% mutate(MI = 1)) %>%
            left_join(Ischemic_Stroke %>% mutate(Ischemic_Stroke = 1)) %>%
            left_join(Heart_Failure %>% mutate(Heart_Failure = 1) %>% select(-standard_concept_name)) %>%
            mutate(MI = coalesce(MI, 0),
                   Ischemic_Stroke = coalesce(Ischemic_Stroke, 0),
                   Heart_Failure = coalesce(Heart_Failure, 0),
                   CVD1_first = pmin(Ischemic_Stroke_first, MI_first, na.rm=T),                 
                   CVD1 = as.numeric(!is.na(CVD1_first)),
                   CVD2_first = pmin(Ischemic_Stroke_first, MI_first, Heart_Failure_first, na.rm=T),                 
                   CVD2 = as.numeric(!is.na(CVD2_first))
                  ) 
names(CVD_summary)

# Finalize output datasets

## All patients

In [ ]:
outcomes <- fullcohortIDs %>%
    left_join(DM_summary %>% mutate(DiabetesMellitus = 1)) %>%
    mutate(DiabetesMellitus = coalesce(DiabetesMellitus, 0)) %>%
    left_join(kidney_disease) %>%
    left_join(CVD_summary) 

Get the percent missingness for each field:

In [ ]:
apply(outcomes, 2, function(x) { signif(100*mean(is.na(x)), 3) })

In [ ]:
today_string <- gsub("-", "", lubridate::today())
today_string
write_to_bucket(outcomes, paste0("outcomes_all_participants_", today_string, ".csv")) 

## Generate Diabetes-specific Dataset

### Add Diabetic Retinopathy (not possible for non-DM)

In [ ]:
#DR1 is only diabetic retinopathy. 
DR1_df <- pull_condition("4174977", name = "DR1")
dim(DR1_df)

In [ ]:
#DR2 is any diabetic eye disease. 
DR2_df <- pull_condition("443767", name = "DR2")
dim(DR2_df)

In [ ]:
DR2_df %>% count(standard_concept_name, sort=T)

In [ ]:
DR1_first <- DR1_df %>%
    group_by(person_id) %>%
    summarize(DR1_first = min(condition_start_date, na.rm=T),
             standard_concept_name = paste0(standard_concept_name, collapse = " / "))
nrow(DR1_first)

In [ ]:
DR2_first <- DR2_df %>%
    group_by(person_id) %>%
    summarize(DR2_first = min(condition_start_date, na.rm=T),
             standard_concept_name = paste0(standard_concept_name, collapse = " / "))
nrow(DR2_first)

### Define the outcomes and add time-to-event from DMDx to event time for all events. 

In [ ]:
last_visit <- read_cols("covariates_wide_all_participants.csv", 
                        select = c("person_id", "last_outpat_or_home_visit", "last_visit_any"), skip_copy = TRUE)

mean(is.na(last_visit$last_outpat_or_home_visit))
mean(is.na(last_visit$last_visit_any))

In [ ]:
outcomes_diabetics <- outcomes %>% filter(DiabetesMellitus == 1)
names(outcomes_diabetics) <- gsub("CKD", "DKD", names(outcomes_diabetics))
outcomes_diabetics <- outcomes_diabetics %>%
    left_join(DR1_first %>% select(-standard_concept_name)) %>%
    left_join(DR2_first %>% select(-standard_concept_name)) %>%
    left_join(last_visit) %>%
    select(-last_visit_any) %>%
    filter(!is.na(last_outpat_or_home_visit)) %>%
    mutate(years_from_dm_to_censor = decimal_date(last_outpat_or_home_visit) - decimal_date(DMDx_first)) %>%
    mutate(DR1 = as.numeric(!is.na(DR1_first)),
          DR2 = as.numeric(!is.na(DR1_first))) %>%
    mutate(years_dm_to_DR1 = coalesce(
        decimal_date(DR1_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_DR2 = coalesce(
        decimal_date(DR2_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_DKD4 = coalesce(
        decimal_date(DKD4_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_RenalFailure = coalesce(
        decimal_date(RenalFailure_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_CVD1 = coalesce(
        decimal_date(CVD1_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_CVD2 = coalesce(
        decimal_date(CVD2_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_MI = coalesce(
        decimal_date(MI_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_Ischemic_Stroke = coalesce(
        decimal_date(Ischemic_Stroke_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    mutate(years_dm_to_Heart_Failure = coalesce(
        decimal_date(Heart_Failure_first) - decimal_date(DMDx_first),
        years_from_dm_to_censor)
    )  %>%
    #reorder
    select(person_id, contains("DM", ignore.case = FALSE), 
           contains(c("DKD", "RenalFailure", "CVD1", "CVD2")), 
           contains("MI", ignore.case=FALSE),
           contains(c("Ischemic", "Heart_Failure", "DR1", "DR2", "EGFR", 
                      "albumin", "Dialysis", "xplant", "last", "year", "survey", "AnyRF")))

In [ ]:
nrow(outcomes_diabetics)

In [ ]:
apply(outcomes_diabetics, 2, function(x) { signif(100*mean(is.na(x)), 3) })

In [ ]:
today_string <- gsub("-", "", lubridate::today())
today_string
write_to_bucket(outcomes_diabetics, paste0("outcomes_diabetics_", today_string, ".csv")) 

# Clean up

In [ ]:
toc()

In [ ]:
today_string <- gsub("-", "", lubridate::today())
today_string

save.image(paste0("outcomes_allpatients_checkpoint_", today_string, ".RData"))

In [ ]:
rm(list = ls())
gc()